# Predictive System Using Crystal Graph Convolution Neural Network

In [ ]:
# predict.py
# ----------
# Inference pipeline for the trained CGCNN model: takes a raw crystal
# structure (a .cif file, any format ase.io.read can open, or an existing
# ase.Atoms object) and outputs the predicted log10(G_vrh) — and the
# corresponding shear modulus in GPa.

# Usage
# -----
#     from predict import CGCNNPredictor

#     predictor = CGCNNPredictor(
#         model_path="elastic-model.pt",
#         node_feat_dim=100,      # must match training (MAX_ATOMIC_NUMBER)
#         edge_feat_dim=40,       # must match training (num_gaussians)
#         cutoff=4,               # must match training
#     )

#     result = predictor.predict("my_structure.cif")
#     print(result)
#     # {'formula': 'Ag2CaGe2', 'log10_G_vrh': 1.42, 'G_vrh_GPa': 26.3}

#     # Also works directly on an ase.Atoms object you already have in memory:
#     from ase import Atoms
#     atoms = Atoms(...)
#     result = predictor.predict(atoms)

#     # Or predict on a whole batch of structures at once (more efficient
#     # than calling .predict() in a loop, since it's a single forward pass):
#     results = predictor.predict_batch(["a.cif", "b.cif", "c.cif"])
# ------------------------------
# Gather structure (.cif) file(s), 'bulk_mod_dataset.json', elastic-model.pt, new_utils2.py


### Installing Required Libraries

In [ ]:
# Installing required packages
# To ensure compatibility with PyTorch Geometric's pre-compiled wheels for CUDA 12.x in Colab,
# it's often necessary to install a specific, known-working PyTorch version.
# We will install PyTorch 2.3.0 with CUDA 12.1 (which works with Colab's CUDA 12.8 drivers).
!pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121

# Install torch-scatter and torch-sparse (these will likely install numpy 2.x and scipy 1.18.0 initially)
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.3.0+cu121.html --force-reinstall

# Install other dependencies like matminer, pymatgen, ase
!pip install matminer pymatgen ase

# Fix for numpy/scipy incompatibility: Forcefully downgrade numpy and scipy LAST
# This addresses the ImportError: cannot import name '_center' from 'numpy._core.umath'
# by ensuring that numpy 1.x and a compatible scipy version are the final installed versions.
# We'll use scipy==1.17.0 as it's the minimum required by pymatgen and is compatible with numpy 1.x.
!pip install numpy==1.26.4 scipy==1.17.0 --force-reinstall


Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.9/780.9 MB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 25.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 103.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 70.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 29.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 73.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/19

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 798.4 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.1/829.1 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━

### Converting Atoms To Data

In [ ]:
import os
import ase.io
import numpy as np
import torch
from ase import Atoms
from ase.neighborlist import neighbor_list
from new_utils3 import CGCNN, Data, GaussianExpansion, MAX_ATOMIC_NUMBER, collate_fn, MaterialsDataset, plot_sample


def _atoms_to_data(atoms, cutoff, gdf):
    """
    Converts a single ase.Atoms object into a Data object, using the exact
    same graph-construction steps MaterialsDataset used during training:
    periodic neighbor search + one-hot node features + gaussian-expanded
    edge distances.
    """
    edge_src, edge_dst, edge_len = neighbor_list("ijd", atoms, cutoff)

    node_feat = np.zeros((len(atoms), MAX_ATOMIC_NUMBER), dtype=np.float32)
    for k, z in enumerate(atoms.numbers):
        node_feat[k, int(z) - 1] = 1.0

    edge_feat = gdf.expand(edge_len)

    return Data(
        node_feat=torch.tensor(node_feat, dtype=torch.float32),
        edge_feat=torch.tensor(edge_feat, dtype=torch.float32),
        edge_src=torch.LongTensor(edge_src),
        edge_dst=torch.LongTensor(edge_dst),
        # target is unused at inference time (nothing reads it in model.forward),
        # but Data/Batch/collate_fn expect the field to be present.
        target=torch.tensor([0.0], dtype=torch.float32),
        atoms=atoms,
    )

### Loading The Structure

In [ ]:
def _load_structure(structure):
    """Accepts a file path (str) or an ase.Atoms object; returns an ase.Atoms."""
    if isinstance(structure, Atoms):
        return structure
    if isinstance(structure, (str, os.PathLike)):
        return ase.io.read(structure)
    raise TypeError(
        f"Expected a file path or an ase.Atoms object, got {type(structure)}."
    )

### Defining CGCNN Predictor Class

In [102]:
class CGCNNPredictor:
    """
    Wraps a trained CGCNN model for easy inference on new crystal structures.

    Parameters
    ----------
    model_path : str
        Path to the saved model weights (e.g. "elastic-model.pt", saved via
        torch.save(model.state_dict(), ...) during training).
    node_feat_dim, edge_feat_dim, node_hidden_dim, num_graph_conv_layers,
    fc_feat_dim : int
        Must match the architecture used during training.
    cutoff : float
        Neighbor-search cutoff radius (Å). Must match training.
    num_gaussians : int
        Number of gaussians in the edge embedding. Must match training.
    device : str
        "cpu" or "cuda".
    """

    def __init__(
        self,
        model_path,
        node_feat_dim=MAX_ATOMIC_NUMBER,
        edge_feat_dim=40,
        node_hidden_dim=64,
        num_graph_conv_layers=3,
        fc_feat_dim=128,
        cutoff=4,
        num_gaussians=40,
        device=None,
    ):
        self.cutoff = cutoff
        self.gdf = GaussianExpansion(cutoff, num_gaussians)
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        self.model = CGCNN(
            node_feat_dim=node_feat_dim,
            edge_feat_dim=edge_feat_dim,
            node_hidden_dim=node_hidden_dim,
            num_graph_conv_layers=num_graph_conv_layers,
            fc_feat_dim=fc_feat_dim,
        )
        state_dict = torch.load(model_path, map_location=self.device)
        self.model.load_state_dict(state_dict)
        self.model.to(self.device)
        self.model.eval()  # inference mode: stable BatchNorm stats, no dropout

    @torch.no_grad()  # disables gradient tracking for everything in this method
    def predict_batch(self, structures):
        """
        Predicts log10(G_vrh) for a list of structures in a single forward
        pass (more efficient than calling .predict() one at a time, since
        the graphs are batched together like during training).

        Parameters
        ----------
        structures : list of (str | ase.Atoms)

        Returns
        -------
        list of dict, one per input structure, each with:
            "formula", "log10_G_vrh", "G_vrh_GPa"
        """
        atoms_list = [_load_structure(s) for s in structures]
        data_list = [_atoms_to_data(a, self.cutoff, self.gdf) for a in atoms_list]

        batch = collate_fn(data_list)
        batch.to(self.device)

        y_pred = self.model(batch)  # shape [num_structures, 1]
        log_vals = y_pred.cpu().view(-1).tolist()

        return [
            {
                "formula": atoms.get_chemical_formula(empirical=True),
                "log10_G_vrh": log_val,
                "G_vrh_GPa": 10 ** log_val,
            }
            for atoms, log_val in zip(atoms_list, log_vals)
        ]

    def predict(self, structure, plot=True, dataset=None):
        """
        Predicts log10(G_vrh) for a single structure.

        Parameters
        ----------
        structure : str | ase.Atoms
        plot : bool, default True
            If True, shows the structure+graph plot (same style as
            utils.plot_sample) before returning the prediction, so you can
            visually confirm what's actually being fed to the model.
        dataset : MaterialsDataset, optional
            If given, checks whether `structure` is already present in this
            dataset (e.g. your training set) and prints a warning if so -
            see is_in_training_set() for details. The result is included in
            the returned dict as "in_training_set" / "training_match_index".

        Returns
        -------
        dict with "formula", "log10_G_vrh", "G_vrh_GPa", and (if `dataset`
        was given) "in_training_set" and "training_match_index".
        """
        atoms = _load_structure(structure)

        # if plot:
        #     edge_src, edge_dst, _ = neighbor_list("ijd", atoms, self.cutoff)
        #     plot_structure(atoms, edge_src, edge_dst, cutoff=self.cutoff)

        result = self.predict_batch([atoms])[0]

        if dataset is not None:
            match = is_in_training_set(atoms, dataset)
            result["in_training_set"] = match["in_training_set"]
            result["training_match_index"] = match["index"]
            if match["in_training_set"]:
                print(
                    f"⚠️  This structure matches training entry "
                    f"#{match['index']} (formula {match['formula']}, via "
                    f"{match['method']} matching). Its prediction reflects "
                    f"memorization, not generalization - pick an unseen "
                    f"structure for a fair test of the model."
                )

        return result

## Predicting

Try some examples, Mn₂CrCo, Ac₂GePd, Ac₂ZnGe, AcTlTe2, Ag2CO3, Al2HgTe4, Ag3Te2Au, AcPbAu2

#### Defining dataset


In [ ]:
# Defining dataset
dataset = MaterialsDataset(
    filename = "/content/bulk_mod_dataset.json",
    cutoff = 4, # cutoff radius is the maximum distance within which two atoms are considered neighbors (Å).
    num_gaussians = 40 # number of gaussians in edge embedding
)

#### Checking The Crystal Structure Whether In The Training Data Or Not

In [81]:
# Defining checking function
def is_in_dataset(structure_path, dataset, tol=0.1):
    atoms = ase.io.read(structure_path)
    formula = atoms.get_chemical_formula()
    vol_per_atom = atoms.get_volume() / len(atoms)
    index = None
    for i, entry in enumerate(dataset.data):
        d_atoms = entry["atoms"]
        if d_atoms.get_chemical_formula() != formula:
            continue
        d_vol_per_atom = d_atoms.get_volume() / len(d_atoms)
        if abs(vol_per_atom - d_vol_per_atom) < tol:
            return True
        index = i
        plot_sample(dataset, idx = i) # Plotting the structure
    return False

# Checking
if is_in_dataset("/content/Mn2CrCo.cif", dataset):
    print("Already in training data")
else:
    print("\nThis structure is not in training data")


This structure is not in training data


#### Predicting The Value

In [103]:
# if __name__ == "__main__":
    # Minimal demo / smoke test. Edit the paths below to match your files.
predictor = CGCNNPredictor(
    model_path="/content/elastic-model-2.pt",
    node_feat_dim=MAX_ATOMIC_NUMBER,
    edge_feat_dim=40,
    cutoff=4,
    num_gaussians=40,
)
result = predictor.predict("/content/Mn2CrCo.cif")
print(result)

{'formula': 'CoCrMn2', 'log10_G_vrh': 1.8825225830078125, 'G_vrh_GPa': 76.29965654151324}


In [ ]:
# Predicted by elastic-model.pt
# 'G_vrh_GPa': 14.161417127309992} for Ac2GePd
# 'G_vrh_GPa': 17.95714519434598} for Ac2GeZn
# 'G_vrh_GPa': 60.14030490152791} for Mn2CrCo
# 'G_vrh_GPa': 12.912056325309477} for AcTlTe2
# 'G_vrh_GPa': 15.29658597393988} for Ag2CO3
# 'G_vrh_GPa': 11.450851205867137} for Al2HgTe4
# 'G_vrh_GPa': 10.82862803236336} for Ag3Te2Au
# 'G_vrh_GPa': 9.441043341969838} for AcPbAu2
#


# Actual values from material project
# 'G_vrh_GPa': 17} for Ac2GePd
# 'G_vrh_GPa': 21} for Ac2GeZn
# 'G_vrh_GPa': 94} for Mn2CrCo
# 'G_vrh_GPa': 20} for AcTlTe2
# 'G_vrh_GPa': 14} for Ag2CO3
# 'G_vrh_GPa': 16} for Al2HgTe4
# 'G_vrh_GPa': 10} for Ag3Te2Au
# 'G_vrh_GPa': 17} for AcPbAu2

In [95]:
# Predicted by elastic-model-2.pt (Best version)
# 'G_vrh_GPa': 16.69210289189864} for Ac2GePd
# 'G_vrh_GPa': 20.56260651946395} for Ac2GeZn
# 'G_vrh_GPa': 76.29965654151324} for Mn2CrCo
# 'G_vrh_GPa': 12.95689400478357} for AcTlTe2
# 'G_vrh_GPa': 16.282065569566793} for Ag2CO3
# 'G_vrh_GPa': 12.581399184153442} for Al2HgTe4
# 'G_vrh_GPa': 10.82862803236336} for Ag3Te2Au
# 'G_vrh_GPa': 11.935100918503485} for AcPbAu2
#


# Actual values from material project
# 'G_vrh_GPa': 17} for Ac2GePd
# 'G_vrh_GPa': 21} for Ac2GeZn
# 'G_vrh_GPa': 94} for Mn2CrCo
# 'G_vrh_GPa': 20} for AcTlTe2
# 'G_vrh_GPa': 14} for Ag2CO3
# 'G_vrh_GPa': 16} for Al2HgTe4
# 'G_vrh_GPa': 10} for Ag3Te2Au
# 'G_vrh_GPa': 17} for AcPbAu2

In [96]:
# Predicted by elastic-model-3.pt
# 'G_vrh_GPa': 19.362186674531017} for Ac2GePd
# 'G_vrh_GPa': 22.1748758235787} for Ac2GeZn
# 'G_vrh_GPa': 69.62328062278232} for Mn2CrCo
# 'G_vrh_GPa': 14.485762563158636} for AcTlTe2
# 'G_vrh_GPa': 20.981427904209976} for Ag2CO3
# 'G_vrh_GPa': 13.811862065751454} for Al2HgTe4
# 'G_vrh_GPa': 12.377406978201885} for Ag3Te2Au
# 'G_vrh_GPa': 12.32541967834595} for AcPbAu2
#


# Actual values from material project
# 'G_vrh_GPa': 17} for Ac2GePd
# 'G_vrh_GPa': 21} for Ac2GeZn
# 'G_vrh_GPa': 94} for Mn2CrCo
# 'G_vrh_GPa': 20} for AcTlTe2
# 'G_vrh_GPa': 14} for Ag2CO3
# 'G_vrh_GPa': 16} for Al2HgTe4
# 'G_vrh_GPa': 10} for Ag3Te2Au
# 'G_vrh_GPa': 17} for AcPbAu2